In [0]:
from pyspark.sql import Row
data= [Row(id=1,name="Arun"),Row(id=2,name="vignesh")]
df=spark.createDataFrame(data)
df.show()

df.write.format("delta").mode("overwrite").save("dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta")

In [0]:
spark.sql("DESCRIBE HISTORY delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`").display()

In [0]:
#insert new records:
from pyspark.sql import Row
data= [Row(id=3,name="Danu"),Row(id=4,name="Rutu")]
df=spark.createDataFrame(data)
df.show()

df.write.format("delta").mode("append").save("dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta")

In [0]:
#using df:
from delta.tables import DeltaTable
from pyspark.sql.functions import col,lit

# Load the Delta table (by catalog name)
delta_table = DeltaTable.forPath(spark, "dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta")

# Perform the update
delta_table.update(
    condition = col("id") == 3,
    set = { "name": lit("Danvik") }
)
spark.sql("""
UPDATE delta.`/Volumes/databricks_practice/inputdb/employee/employee_delta`
SET name = 'danvik'
WHERE id = 3
""")

spark.sql("select * from delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`").display()

In [0]:
spark.sql("DESCRIBE HISTORY delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`").display()

In [0]:
delta_table.delete( condition= "id =2")

spark.sql("select * from delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`").display()

In [0]:
spark.sql("DESCRIBE HISTORY delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`").display()

In [0]:
%sql
insert into delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`(id,name)
values("6","Nand")

In [0]:
%sql
Update delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`set name = "Nandhu"
where id =6

In [0]:
%sql
delete from delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`
where id =6

In [0]:
spark.sql("DESCRIBE HISTORY delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta`").display()

In [0]:
%sql
select * from delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta` version as of 0

In [0]:
df_v2 = spark.read.format("delta").option("versionAsOf", 2).load("dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta")
df_v2.display()


In [0]:
%sql
select * from delta.`dbfs:/Volumes/databricks_practice/inputdb/employee/employee_delta` timestamp as of '2025-10-15 02:16:52'

| Operation  | What Delta Does                                                                                        | Where Info Stored                             |
| ---------- | ------------------------------------------------------------------------------------------------------ | --------------------------------------------- |
| **INSERT** | Adds new Parquet files                                                                                 | `_delta_log/add`                              |
| **UPDATE** | Reads old files with matching rows → writes new files with updated data → marks old files as removed   | `_delta_log/add` + `_delta_log/remove`        |
| **DELETE** | Reads old files with matching rows → writes new files with remaining data → marks old files as removed | `_delta_log/add` + `_delta_log/remove`        |
| **READ**   | Reads the latest valid snapshot (based on `_delta_log` commits)                                        | `_delta_log/latest` (transaction log history) |

